# Build Your Own Data Structures

Python's lists and dicts are wonderful, but they are *someone else's* data structures. In this lab you build the classics yourself — linked list, stack, queue, binary search tree, and heap — because implementing a structure is the only way to truly own its trade-offs. Each one follows the same rhythm: **implement → demo → micro-exercise**, and the lab ends with a bigger set of exercises.

**How to use this notebook:** run cells top to bottom (`Shift+Enter`); later sections reuse earlier classes. The micro-exercise scaffolds run as-is — fill them in as you go, or on a second pass.

Prerequisite: you should be comfortable with classes (`__init__`, `self`, methods) from the collections & OOP notebook.

## 1. Singly linked list

A **linked list** stores items in separate *node* objects, each holding a value and a reference to the **next** node — like a treasure hunt where every clue points to the following one. The list itself only remembers where the chain starts (the **head**); the final node points to `None`.

Compare with a Python list, which keeps elements side by side in one block of memory. The chain layout means a linked list can splice a new first element in **constant time** — $O(1)$ — no shifting required, which is exactly what arrays are bad at.

First, the node — and a chain wired up by hand so you can see there is no magic:

In [ ]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None          # not linked to anything... yet

# Wire up  "to" -> "be" -> "or"  manually:
first = Node("to")
second = Node("be")
third = Node("or")
first.next = second
second.next = third

# Walk the chain: start at the front, follow .next until None
current = first
while current is not None:
    print(current.value, end=" -> ")
    current = current.next
print("None")

That `while current is not None: ... current = current.next` walk is *the* fundamental linked-list move — every operation below is a variation of it.

Wiring nodes by hand is tedious, so we wrap the bookkeeping in a class. `prepend` shows off the $O(1)$ party trick; `append` must walk to the end first, so it costs $O(n)$:

In [ ]:
class SinglyLinkedList:
    def __init__(self):
        self.head = None

    def prepend(self, value):         # O(1): re-point head, done
        node = Node(value)
        node.next = self.head
        self.head = node

    def append(self, value):          # O(n): must walk to the last node
        node = Node(value)
        if self.head is None:
            self.head = node
            return
        last = self.head
        while last.next is not None:
            last = last.next
        last.next = node

    def to_list(self):                # collect values into a Python list
        values, current = [], self.head
        while current is not None:
            values.append(current.value)
            current = current.next
        return values

Demo time — watch where each value lands:

In [ ]:
chain = SinglyLinkedList()
chain.append("B")
chain.append("C")
chain.prepend("A")        # jumps the queue, instantly
chain.append("D")

print(chain.to_list())
assert chain.to_list() == ["A", "B", "C", "D"]
print("Chain is wired correctly.")

### Micro-exercise: `contains`

Write `ll_contains(linked, target)` that walks the chain and returns `True` if any node holds `target`. It's the traversal loop with an `if` inside.

In [ ]:
def ll_contains(linked, target):
    current = linked.head
    # your code here: walk the chain, return True on a match
    return False

# Uncomment to test:
# assert ll_contains(chain, "C") == True
# assert ll_contains(chain, "Z") == False
# print("ll_contains works!")

## 2. Stack — and the balanced-brackets problem

A **stack** is a last-in-first-out (**LIFO**) container: like a stack of plates, you may only `push` onto the top and `pop` off the top. Undo history, the browser back button, and function calls all run on stacks.

A Python list already does everything we need (append/pop at the end are both $O(1)$), so our class is a thin, honest wrapper that *enforces the discipline* — no sneaky access to the middle:

In [ ]:
class Stack:
    def __init__(self):
        self._items = []              # leading _ means: internal, hands off

    def push(self, item):
        self._items.append(item)

    def pop(self):
        return self._items.pop()      # removes and returns the TOP item

    def peek(self):
        return self._items[-1]        # look at the top without removing

    def is_empty(self):
        return len(self._items) == 0

s = Stack()
for plate in ["red", "green", "blue"]:
    s.push(plate)
print(s.pop(), s.pop(), s.pop())      # comes back reversed!

Pushed red-green-blue, popped blue-green-red: a stack *reverses* whatever passes through it. Remember that — it's exercise fuel.

**The classic application: balanced brackets.** Is `([{}])` well-formed? Is `([)]`? The rule: every closer must match the *most recently opened* bracket — "most recent" is precisely what a stack tracks. Push every opener; on each closer, pop and check it matches; at the end the stack must be empty.

In [ ]:
def is_balanced(text):
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = Stack()
    for ch in text:
        if ch in "([{":
            stack.push(ch)
        elif ch in ")]}":
            if stack.is_empty() or stack.pop() != pairs[ch]:
                return False          # closer with no/wrong partner
    return stack.is_empty()           # leftovers mean unclosed openers

for candidate in ["([{}])", "([)]", "(()", "x = arr[f(2)]", ")("]:
    print(f"{candidate:>15}  ->  {is_balanced(candidate)}")

Check `([)]` by hand: push `(`, push `[`, then `)` arrives — pop gives `[`, which doesn't match. Rejected, exactly as your code editor's bracket checker would. Note the three ways to fail: wrong partner, a closer on an empty stack (`)(`), and leftover openers (`((`).

### Micro-exercise: reverse a string with a stack

Use the reversing superpower deliberately: push every character of a string, then pop them all into a new string.

In [ ]:
def reverse_string(text):
    stack = Stack()
    result = ""
    # your code here: push all characters, then pop while not empty
    return result

# Uncomment to test:
# assert reverse_string("stressed") == "desserts"
# assert reverse_string("") == ""
# print("reverse_string works!")

## 3. Queue — via `deque`

A **queue** is first-in-first-out (**FIFO**): the supermarket line. Add at the back (**enqueue**), remove from the front (**dequeue**).

Why not just use a list? `list.pop(0)` must shift every remaining element left — $O(n)$ per dequeue, brutally slow for long queues. The standard library's `collections.deque` ("double-ended queue", pronounced *deck*) removes from either end in $O(1)$, so it's the professional choice:

In [ ]:
from collections import deque

line = deque()
line.append("Ana")        # enqueue at the back
line.append("Ben")
line.append("Chi")

print(line.popleft())     # dequeue from the front: first in, first out
print(line.popleft())
print("still waiting:", list(line))

**Demo: a shared printer.** Jobs arrive in order and the printer handles them one at a time. How long does each person wait before their pages even *start* printing? A queue simulation answers it — the same math schedules real print servers, help desks, and CPU tasks.

In [ ]:
import random
random.seed(9)                        # seeded, so everyone sees the same run

jobs = deque()
for owner in ["Ana", "Ben", "Chi", "Dmitri"]:
    jobs.append((owner, random.randint(1, 8)))   # (owner, pages)

print("queue:", list(jobs), "\n")
clock = 0                             # minutes; 1 page = 1 minute
while jobs:
    owner, pages = jobs.popleft()
    print(f"t={clock:>2}  {owner}'s job starts ({pages} pages, waited {clock} min)")
    clock += pages
print(f"t={clock:>2}  printer idle")

Everyone's waiting time is the total of the jobs *ahead* of them — which is why one 8-page job early in the queue ruins everyone's morning, and why real systems sometimes let short jobs jump ahead.

### Micro-exercise: total waiting time

Compute the *sum* of all four waiting times in a fresh copy of the same simulation. (Wait = the clock value when the job starts.) Accumulate instead of printing.

In [ ]:
def total_wait(job_list):
    jobs = deque(job_list)
    clock = 0
    total = 0
    # your code here: pop jobs; add the current clock to total; advance clock
    return total

# Uncomment to test:
# assert total_wait([("A", 3), ("B", 2), ("C", 4)]) == 0 + 3 + 5   # = 8
# print("total_wait works!")

## 4. Binary search tree

A **binary search tree** (BST) is a linked structure where each node has up to two children, arranged by one sacred rule — the **BST invariant**: everything in a node's *left* subtree is smaller than the node's value; everything in the *right* subtree is larger.

The payoff: to find a value you never search the whole tree — at each node you go left or right, discarding the other half, just like binary search on a sorted list. In a reasonably balanced tree that's $O(\log n)$ per lookup.

Insertion follows the same compass: walk down as if searching, and plant the new node at the empty spot where the search falls off the tree.

In [ ]:
class TreeNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

def insert(root, value):
    if root is None:                  # fell off the tree: plant here
        return TreeNode(value)
    if value < root.value:
        root.left = insert(root.left, value)
    elif value > root.value:
        root.right = insert(root.right, value)
    return root                       # (duplicates are simply ignored)

root = None
for v in [8, 3, 10, 1, 6, 14, 4, 7]:
    root = insert(root, v)
print("tree built - root holds:", root.value)
print("root's children:", root.left.value, "and", root.right.value)

Because 8 arrived first it became the root; 3 went left (smaller), 10 went right (larger), and so on recursively. The tree now looks like this:

```text
        8
      /   \
     3     10
    / \      \
   1   6     14
      / \
     4   7
```

Searching is the same walk, minus the planting:

In [ ]:
def search(root, value):
    if root is None:
        return False                  # ran out of tree: not here
    if value == root.value:
        return True
    if value < root.value:
        return search(root.left, value)
    return search(root.right, value)

for probe in [6, 14, 5, 8, 99]:
    print(f"search({probe:>2}) -> {search(root, probe)}")

Finding 6 took three steps (8 → 3 → 6) instead of looking at all eight values. On a million-node balanced tree the walk is about 20 steps — that's $\log_2$ at work.

**The bonus miracle: in-order traversal.** Visit each node's left subtree, then the node, then its right subtree — recursively. The BST invariant guarantees the values come out **sorted**:

In [ ]:
def in_order(root, out=None):
    if out is None:
        out = []
    if root is not None:
        in_order(root.left, out)      # 1. everything smaller
        out.append(root.value)        # 2. me
        in_order(root.right, out)     # 3. everything larger
    return out

values = in_order(root)
print(values)
assert values == sorted(values)
print("In-order traversal is sorted - the invariant delivers!")

### Micro-exercise: count the nodes

Write `count_nodes(root)`: an empty tree has 0 nodes; otherwise it's 1 + the count of the left subtree + the count of the right subtree. Three lines of recursion.

In [ ]:
def count_nodes(root):
    # your code here
    pass

# Uncomment to test:
# assert count_nodes(root) == 8
# assert count_nodes(None) == 0
# print("count_nodes works!")

## 5. Min-heap — via `heapq`

A **min-heap** is a structure with one obsession: hand me the smallest item, *fast*. Push in $O(\log n)$, pop-the-smallest in $O(\log n)$, and peeking at the smallest is free. It powers priority queues (emergency rooms, not supermarket lines: most urgent first, not first-come-first-served).

Python ships it as the `heapq` module, which treats a plain list as a heap:

In [ ]:
import heapq
import random
random.seed(4)

arrivals = [random.randint(1, 99) for _ in range(8)]
print("arrival order:", arrivals)

heap = []
for x in arrivals:
    heapq.heappush(heap, x)

drained = [heapq.heappop(heap) for _ in range(len(arrivals))]
print("popped order: ", drained)      # always smallest-first!

Values went in scrambled and came out ascending — the heap maintains just enough internal order to always know its minimum. (Peek at the raw `heap` list before draining sometime: it is *not* fully sorted; that's the trick that keeps it fast.)

**The killer app: top-k.** "Largest 3 of these 1000 scores" doesn't need a full sort — `heapq.nlargest` / `nsmallest` do it directly, and internally they keep a heap of only $k$ items:

In [ ]:
import heapq
import random
random.seed(4)

scores = [random.randint(0, 1000) for _ in range(1000)]

print("top 3 scores:   ", heapq.nlargest(3, scores))
print("bottom 3 scores:", heapq.nsmallest(3, scores))

# Cross-check against a full sort:
assert heapq.nlargest(3, scores) == sorted(scores, reverse=True)[:3]
print("Matches the full sort - with far less work on big data.")

### Micro-exercise: third-smallest

Using `heappush` and `heappop` yourself (no `nsmallest`!), write `third_smallest(items)`: push everything onto a heap, pop twice, and return the third pop.

In [ ]:
def third_smallest(items):
    heap = []
    # your code here: push all items, pop two, return the third pop
    pass

# Uncomment to test:
# assert third_smallest([50, 20, 90, 10, 70]) == 50
# assert third_smallest([3, 1, 2]) == 3
# print("third_smallest works!")

## Choosing your structure

| Structure | Superpower | Watch out |
|---|---|---|
| linked list | $O(1)$ insert at front | $O(n)$ to reach position $i$ |
| stack (LIFO) | undo / matching / reversal | only the top is reachable |
| queue (FIFO, `deque`) | fairness, buffering | no random access |
| BST | $O(\log n)$ search + sorted traversal | degrades to $O(n)$ if unbalanced |
| min-heap | instant minimum, top-k | only the min is visible |

## Try it yourself

Bigger exercises now — the scaffolds all run as-is.

### Exercise 1 — Reverse a linked list's values

Write `reversed_values(linked)` that returns the values of a `SinglyLinkedList` in back-to-front order — *without* calling `to_list` and reversing it. Hint: walk the chain and **insert each value at position 0** of your result (or push onto a `Stack` and pop).

In [ ]:
def reversed_values(linked):
    result = []
    current = linked.head
    # your code here
    return result

# Uncomment to test (uses `chain` = A, B, C, D from earlier):
# assert reversed_values(chain) == ["D", "C", "B", "A"]
# print("reversed_values works!")

### Exercise 2 — Hot potato

Children stand in a circle passing a potato; after every `passes` passes, whoever holds it is out, and the game continues until one child remains. Simulate the circle with a `deque`: `rotate(-1)` (or popleft+append) passes the potato; `popleft()` eliminates the holder. Return the winner.

In [ ]:
from collections import deque

def hot_potato(names, passes):
    circle = deque(names)
    # your code here: while more than one remains, pass `passes` times,
    # then eliminate the holder at the front
    return circle[0]

# Uncomment to test (eliminations go C, A, E, B - check by hand!):
# assert hot_potato(["A", "B", "C", "D", "E"], 2) == "D"
# print("winner:", hot_potato(["Ana", "Ben", "Chi", "Dmitri"], 7))

### Exercise 3 — Smallest value in a BST

The BST invariant makes one query trivial: the minimum is the **leftmost** node. Write `bst_minimum(root)` — keep stepping `.left` until you can't. (What should happen on an empty tree? Decide and document with a comment.)

In [ ]:
def bst_minimum(root):
    # your code here
    pass

# Uncomment to test (uses `root` from the BST section - minimum is 1):
# assert bst_minimum(root) == 1
# print("bst_minimum works!")

### Exercise 4 — Merge two sorted lists with a heap

Write `merge_sorted(a, b)` that merges two already-sorted lists into one sorted list: push everything from both onto a heap, then pop until empty. Then ponder: could you do it *without* a heap, walking both lists with two fingers? (That version is the heart of merge sort, coming up in the algorithms lab.)

In [ ]:
import heapq

def merge_sorted(a, b):
    merged = []
    # your code here
    return merged

# Uncomment to test:
# assert merge_sorted([1, 4, 9], [2, 3, 10]) == [1, 2, 3, 4, 9, 10]
# assert merge_sorted([], [5]) == [5]
# print("merge_sorted works!")

### Exercise 5 — Design decision drill

No code — write your answers as a comment in the cell below, then run it (comments run happily). For each scenario pick the best structure from this lab and note *why* in a few words:

1. Browser history for a back button.
2. Chat messages waiting to be sent, in order.
3. An autocomplete dictionary needing fast sorted listing of all entries.
4. A game leaderboard that must always show the current top 10.

In [ ]:
# My picks:
# 1. back button      ->  your answer here
# 2. outgoing chat    ->  your answer here
# 3. autocomplete     ->  your answer here
# 4. leaderboard      ->  your answer here
print("Decisions recorded - compare with the table above.")